# 🥐 Bakery Market Basket Analysis — Association Pattern Mining

**Framework:** CRISP-DM  
**Technique:** Frequent itemsets + Association Rules  
**Business use:** Cross-sell, bundles, recommendations, product placement

Dataset: popular Kaggle **Bread Basket / Bakery Sales** transaction data.

This notebook first attempts to download the full public CSV. If internet is unavailable, it falls back to an embedded real excerpt so the notebook remains reproducible.


## 1. Business Understanding

The bakery wants to discover products that tend to occur together in the same transaction.

### Business questions
- Which products dominate baskets?
- Which combinations are frequent?
- Which rules have meaningful **support**, **confidence**, and **lift**?
- Which rules can become bundles, checkout recommendations, or shelf/menu placement actions?

### Success criteria
A useful association rule should combine:
- enough **support** to matter,
- good **confidence**,
- **lift > 1**, indicating positive association.


In [ ]:
import pandas as pd, numpy as np, itertools
import matplotlib.pyplot as plt
from io import StringIO

FULL_DATA_URL = "https://raw.githubusercontent.com/prasertcbs/basic-dataset/refs/heads/master/BreadBasket_DMS.csv"

REAL_FALLBACK = r"""Date,Time,Transaction,Item
2016-10-30,09:58:11,1,Bread
2016-10-30,10:05:34,2,Scandinavian
2016-10-30,10:05:34,2,Scandinavian
2016-10-30,10:07:57,3,Hot chocolate
2016-10-30,10:07:57,3,Jam
2016-10-30,10:07:57,3,Cookies
2016-10-30,10:08:41,4,Muffin
2016-10-30,10:13:03,5,Coffee
2016-10-30,10:13:03,5,Pastry
2016-10-30,10:13:03,5,Bread
2016-10-30,10:16:55,6,Medialuna
2016-10-30,10:16:55,6,Pastry
2016-10-30,10:16:55,6,Muffin
2016-10-30,10:19:12,7,Medialuna
2016-10-30,10:19:12,7,Pastry
2016-10-30,10:19:12,7,Coffee
2016-10-30,10:19:12,7,Tea
2016-10-30,10:20:51,8,Pastry
2016-10-30,10:20:51,8,Bread
2016-10-30,10:21:59,9,Bread
2016-10-30,10:21:59,9,Muffin
2016-10-30,10:25:58,10,Scandinavian
2016-10-30,10:25:58,10,Medialuna
2016-10-30,10:27:21,11,Bread
2016-10-30,10:27:21,11,Medialuna
2016-10-30,10:27:21,11,Bread
2016-10-30,10:27:21,11,NONE
2016-10-30,10:30:14,12,Jam
2016-10-30,10:30:14,12,Coffee
2016-10-30,10:30:14,12,Tartine
2016-10-30,10:30:14,12,Pastry
2016-10-30,10:30:14,12,Tea
2016-10-30,10:31:24,13,Basket
2016-10-30,10:31:24,13,Bread
2016-10-30,10:31:24,13,Coffee
2016-10-30,10:32:46,14,Bread
2016-10-30,10:32:46,14,Medialuna
2016-10-30,10:32:46,14,Pastry
2016-10-30,10:34:36,15,NONE
2016-10-30,10:34:36,15,NONE
2016-10-30,10:34:36,15,Mineral water
2016-10-30,10:34:36,15,Scandinavian
2016-10-30,10:37:08,16,Bread
2016-10-30,10:37:08,16,Medialuna
2016-10-30,10:37:08,16,Coffee
2016-10-30,10:38:04,17,Hot chocolate
2016-10-30,10:41:56,18,Farm House
2016-10-30,10:43:08,19,Farm House
2016-10-30,10:43:08,19,Bread
2016-10-30,10:45:22,20,Bread
2016-10-30,10:45:22,20,Medialuna
2016-10-30,10:49:29,21,Coffee
2016-10-30,10:49:29,21,Coffee
2016-10-30,10:49:29,21,Medialuna
2016-10-30,10:49:29,21,Bread
2016-10-30,10:52:15,22,Jam
2016-10-30,10:53:49,23,Scandinavian
2016-10-30,10:53:49,23,Muffin
2016-10-30,10:54:33,24,Bread
2016-10-30,10:55:22,25,Scandinavian
2016-10-30,10:56:08,26,Fudge
2016-10-30,11:02:19,27,Scandinavian
2016-10-30,11:03:24,28,Coffee
2016-10-30,11:03:24,28,Bread
2016-10-30,11:05:30,29,Bread
2016-10-30,11:05:30,29,Jam
2016-10-30,11:05:30,29,NONE
2016-10-30,11:07:19,30,Bread
2016-10-30,11:12:56,31,Basket
2016-10-30,11:16:15,32,Scandinavian
2016-10-30,11:16:15,32,Muffin
2016-10-30,11:22:49,33,Coffee
2016-10-30,11:25:45,34,Coffee
2016-10-30,11:25:45,34,Muffin
2016-10-30,11:27:34,35,Muffin
2016-10-30,11:27:34,35,Scandinavian
2016-10-30,11:33:08,36,Tea
2016-10-30,11:33:08,36,Bread
2016-10-30,11:37:10,37,Coffee
2016-10-30,11:37:10,37,Bread
2016-10-30,11:37:10,37,NONE
2016-10-30,11:42:40,38,Bread
2016-10-30,11:42:40,38,Tea
2016-10-30,11:44:31,39,Scandinavian
2016-10-30,11:55:51,40,Juice
2016-10-30,11:55:51,40,NONE
2016-10-30,11:55:51,40,Tartine
2016-10-30,11:55:51,40,Coffee
2016-10-30,11:55:51,40,Muffin
2016-10-30,11:57:06,41,Scandinavian
2016-10-30,11:57:45,42,Bread
2016-10-30,11:57:45,42,Tea
2016-10-30,12:00:22,43,Scandinavian
2016-10-30,12:00:22,43,Fudge
2016-10-30,12:05:47,44,Coffee
2016-10-30,12:05:47,44,Medialuna
2016-10-30,12:08:36,45,Coffee
2016-10-30,12:08:36,45,Hot chocolate
2016-10-30,12:08:36,45,Medialuna
2016-10-30,12:09:04,46,Coffee
2016-10-30,12:15:29,47,Ella's Kitchen Pouches
2016-10-30,12:15:29,47,Juice
2016-10-30,12:15:29,47,Bread
2016-10-30,12:15:29,47,Muffin
2016-10-30,12:15:29,47,Jam
2016-10-30,12:17:02,48,Coffee
2016-10-30,12:23:01,49,Coffee
2016-10-30,12:23:01,49,Coffee
2016-10-30,12:23:01,49,Medialuna
2016-10-30,12:25:11,50,Bread
2016-10-30,12:25:11,50,Victorian Sponge
2016-10-30,12:26:58,51,Bread
2016-10-30,12:28:07,52,Scandinavian
2016-10-30,12:33:08,54,Bread
2016-10-30,12:38:06,55,Frittata
2016-10-30,12:38:06,55,Coffee
2016-10-30,12:38:06,55,Tea
2016-10-30,12:38:06,55,Hearty & Seasonal
2016-10-30,12:39:27,56,Coffee
2016-10-30,12:39:27,56,Frittata
2016-10-30,12:45:48,57,Scandinavian
2016-10-30,12:59:29,58,Victorian Sponge
2016-10-30,12:59:29,58,Hot chocolate
2016-10-30,12:59:29,58,Tea
2016-10-30,12:59:29,58,Soup
2016-10-30,13:02:04,59,Tea
2016-10-30,13:02:04,59,NONE
2016-10-30,13:14:04,60,Cookies
2016-10-30,13:14:04,60,Coffee
2016-10-30,13:14:04,60,Juice
2016-10-30,13:14:04,60,Coffee
2016-10-30,13:15:38,61,Coffee
2016-10-30,13:24:00,62,Pick and Mix Bowls
2016-10-30,13:24:00,62,Hearty & Seasonal
2016-10-30,13:24:00,62,Hearty & Seasonal
2016-10-30,13:24:00,62,Coffee
2016-10-30,13:24:00,62,Smoothies
2016-10-30,13:24:00,62,Coffee
2016-10-30,13:33:12,63,Coffee
2016-10-30,13:34:07,64,Cake
2016-10-30,13:37:25,65,NONE
2016-10-30,13:37:25,65,Tartine
2016-10-30,13:37:25,65,Mighty Protein
2016-10-30,13:37:25,65,Tea
2016-10-30,13:37:25,65,Coffee
2016-10-30,13:40:20,66,Hearty & Seasonal
2016-10-30,13:40:20,66,Frittata
2016-10-30,13:40:20,66,Mineral water
2016-10-30,13:46:48,67,Hearty & Seasonal
2016-10-30,13:46:48,67,NONE
2016-10-30,13:46:48,67,Mineral water
2016-10-30,13:46:48,67,Muffin
2016-10-30,13:49:36,68,Frittata
2016-10-30,13:49:36,68,Coffee
2016-10-30,13:49:36,68,Tea
2016-10-30,13:49:36,68,Scandinavian
2016-10-30,13:49:36,68,Chicken sand
2016-10-30,13:51:49,69,Bread
2016-10-30,13:51:49,69,Tea
2016-10-30,13:51:49,69,Victorian Sponge
2016-10-30,13:54:42,70,Fudge
2016-10-30,13:56:08,71,Muffin
2016-10-30,14:21:38,72,Coffee
2016-10-30,14:21:38,72,Bread
2016-10-30,14:22:39,73,Bread
2016-10-30,14:24:03,74,Coffee
2016-10-30,14:24:03,74,Bread
2016-10-30,14:32:26,75,NONE
2016-10-30,14:32:26,75,Jam
2016-10-30,14:32:26,75,Frittata
2016-10-30,14:32:49,76,Coffee
2016-10-30,14:35:36,77,Scandinavian
2016-10-30,14:42:29,78,Fudge
2016-10-30,14:45:44,79,Fudge
2016-10-30,14:45:44,79,Tea
2016-10-30,14:45:44,79,Coffee
2016-10-30,14:45:44,79,Muffin
2016-10-30,14:48:40,80,Frittata
2016-10-30,14:48:40,80,Bread
2016-10-30,14:48:40,80,Coffee
"""

try:
    raw = pd.read_csv(FULL_DATA_URL)
    DATA_MODE = "FULL PUBLIC DATASET"
except Exception:
    raw = pd.read_csv(StringIO(REAL_FALLBACK))
    DATA_MODE = "EMBEDDED REAL-DATA SAMPLE"

print("Mode:", DATA_MODE)
print("Shape:", raw.shape)
display(raw.head())


## 2. Data Understanding

Inspect the transaction structure, missing/sentinel values, item frequency, time coverage, and duplicates.


In [ ]:
print(raw.info())
print("\nMissing values:")
display(raw.isna().sum().to_frame("missing"))

print("\nUnique transactions:", raw["Transaction"].nunique())
print("Unique raw items:", raw["Item"].nunique())
print("Duplicate rows:", raw.duplicated().sum())


## 3. Data Preparation

Cleaning decisions:
- trim item names,
- remove `NONE` placeholder records,
- remove repeated copies of the same item inside the same transaction for binary market-basket analysis,
- parse timestamp,
- create transaction baskets.


In [ ]:
df = raw.copy()
df["Item"] = df["Item"].astype(str).str.strip()
df = df[df["Item"].str.upper().ne("NONE")]
df = df.drop_duplicates(["Transaction","Item"])
df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"])

transactions = df.groupby("Transaction")["Item"].apply(lambda x: sorted(set(x))).to_dict()
baskets = list(transactions.values())

print("Clean rows:", len(df))
print("Transactions:", len(baskets))
print("Unique products:", df["Item"].nunique())
print("Average basket size:", np.mean([len(x) for x in baskets]).round(2))


In [ ]:
item_counts = df["Item"].value_counts().head(15)
display(item_counts.to_frame("transactions"))

plt.figure(figsize=(10,6))
item_counts.sort_values().plot(kind="barh")
plt.title("Top Bakery Items")
plt.xlabel("Transaction appearances")
plt.tight_layout()
plt.show()


## 4. Modeling — Frequent Itemsets & Association Rules

To keep the notebook dependency-light, Apriori-style frequent itemset mining is implemented directly with Python combinations.

We mine itemsets up to size 3, then derive rules and compute:

- **Support:** frequency of A and B occurring together
- **Confidence:** P(B | A)
- **Lift:** confidence divided by baseline probability of B
- **Leverage:** observed co-occurrence minus independence expectation
- **Conviction:** directional implication strength


In [ ]:
def support(itemset):
    target = set(itemset)
    return sum(target.issubset(set(b)) for b in baskets) / len(baskets)

items = sorted(df["Item"].unique())

# Adaptive support works for both the full dataset and fallback sample.
MIN_SUPPORT = 0.01 if len(baskets) > 1000 else 0.04

supports = {}
for k in [1,2,3]:
    for combo in itertools.combinations(items, k):
        sup = support(combo)
        if sup >= MIN_SUPPORT:
            supports[frozenset(combo)] = sup

print("Frequent itemsets:", len(supports))


In [ ]:
rules = []
for itemset, sup_xy in supports.items():
    if len(itemset) < 2:
        continue
    for r in range(1, len(itemset)):
        for ant in itertools.combinations(itemset, r):
            A = frozenset(ant)
            B = itemset - A
            sup_a = supports.get(A, support(A))
            sup_b = supports.get(B, support(B))
            if sup_a == 0 or sup_b == 0:
                continue
            confidence = sup_xy / sup_a
            lift = confidence / sup_b
            leverage = sup_xy - sup_a * sup_b
            conviction = (1-sup_b)/(1-confidence) if confidence < 1 else np.inf
            rules.append({
                "antecedents": ", ".join(sorted(A)),
                "consequents": ", ".join(sorted(B)),
                "support": sup_xy,
                "confidence": confidence,
                "lift": lift,
                "leverage": leverage,
                "conviction": conviction
            })

rules_df = pd.DataFrame(rules)
rules_df = rules_df.sort_values(["lift","confidence","support"], ascending=False)
print("Rules generated:", len(rules_df))
display(rules_df.head(15))


## 5. Evaluation

A large lift from a tiny number of baskets may be unstable. We therefore focus on rules that satisfy both business relevance and statistical frequency.


In [ ]:
strong_rules = rules_df[
    (rules_df["confidence"] >= 0.30) &
    (rules_df["lift"] > 1.05)
].copy()

print("Strong candidate rules:", len(strong_rules))
display(strong_rules.head(15))


In [ ]:
plt.figure(figsize=(9,6))
plt.scatter(rules_df["support"], rules_df["confidence"],
            s=np.clip(rules_df["lift"]*35, 20, 250), alpha=.7)
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.title("Association Rule Quality")
plt.tight_layout()
plt.show()


## 6. Deployment / Business Actions

Association mining is deployed as **decision intelligence**, not a predictive API.

Recommended use:
1. bundle high-lift, adequately-supported pairs;
2. show complementary items in checkout recommendations;
3. place complementary products near each other;
4. monitor rule stability monthly;
5. A/B test promotions before rolling them out broadly.

### Important limitation
Association ≠ causation. Lift identifies co-occurrence beyond chance, not proof that one product causes purchase of another.


# Executive Conclusion

This project completes the CRISP-DM lifecycle:

**Business Understanding → Data Understanding → Data Preparation → Pattern Mining → Evaluation → Business Deployment**

The core deliverable is a ranked set of association rules that translates transaction data into practical cross-sell opportunities.
